## Паплайн для загрузки данных, предобработки, генерации прогнозов и расчета метрик

В данном проекте тестируем доработанную модель с более подробным кодированием ВСЕХ текстовых полей (как при тесте 5) на реальной новой выборке платежей с новыми значениями статей за период с 10.10.2025-04.12.2025. 


**Вывод**: По итогу выловили баг, что тестирование модели на синтетически собранной сбалансированной выборке (тест 5) некорректно, так как в реальных данных сильный дисбаланс классов, поэтому на базе теста 5 переобучили модель на умеренно сбалансированных данных, что в целом повысило метрики именно на реальных данных, как показал этот тест ниже. Эту модель отправим в продакшн.

In [1]:
import sys
import subprocess

# установка и импорт нужной версии PyTorch
try:
       
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "torch==2.6.0",
        "--index-url", "https://download.pytorch.org/whl/cu118"],
        stdout=subprocess.DEVNULL)
except:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "torch==2.6.0"],
        stdout=subprocess.DEVNULL)
try:
    import numpy as np
    if np.__version__ != "1.26.4":
        subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy==1.26.4", "--force-reinstall"])
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy==1.26.4", "--force-reinstall"])
    import numpy as np

# установка и импорт нужной версии spacy и языковой модели
try:
    import spacy
    if spacy.__version__ != "3.6.1":
        subprocess.check_call([sys.executable, "-m", "pip", "install", "spacy==3.6.1", "--force-reinstall"], stdout=subprocess.DEVNULL)
        import importlib
        importlib.reload(spacy)
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "spacy==3.6.1", "--force-reinstall"], stdout=subprocess.DEVNULL)
    import spacy

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "https://github.com/explosion/spacy-models/releases/download/ru_core_news_sm-3.6.0/ru_core_news_sm-3.6.0-py3-none-any.whl"],
    stdout=subprocess.DEVNULL)


import pandas as pd
import numpy as np
import random
import os
import re
import requests
import joblib
import logging

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import IncrementalPCA

# ⏳ прогресс-бары
from tqdm import tqdm

# 🧠 обработка текста и NLP
import spacy
try:
    import transformers
    from transformers import AutoModel, AutoTokenizer
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "transformers"], stdout=subprocess.DEVNULL)
    import transformers
    from transformers import AutoModel, AutoTokenizer    
    
# 🤖 pyTorch
import torch

# загрузка catboost
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier

# загрузка моделей и функци для предобработки данных
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report,roc_auc_score


pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 0)
pd.set_option('display.max_colwidth', 120)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

/opt/anaconda3/envs/datasphere_cli_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

In [3]:
# создаем функцию очистки текста
def _clean_single_text(text):
    return re.sub(r"[^\w\s]", " ", text.lower())

# создаем функцию предобработки текста
def preprocess_texts_optimized(texts, nlp_model_name,
                               batch_size_cpu=256,
                               num_processes_for_cleaning=-1,
                               num_processes_for_spacy_cpu=-1):
    
    print(f"🔍 Запуск предобработки для {len(texts)} текстов...")
    
    # предварительная очистка текстов
    cleaned_texts = [_clean_single_text(text) for text in tqdm(texts, desc="Очистка")]

    # spaCy и лемматизация
    nlp = None
    processed_lemmas = []
    
    # загрузка NLP модели
    print(f"🔍 Загрузка spaCy модели: '{nlp_model_name}'.")
    nlp = spacy.load(nlp_model_name)

    # используем n_process для параллелизации
    if num_processes_for_spacy_cpu == -1:
        cpu_count = os.cpu_count()
        num_processes_for_spacy_cpu = max(1, cpu_count - 1)
    
    print(f"🔍 Лемматизация будет выполнена в {num_processes_for_spacy_cpu} потоках.")
    
    for doc in tqdm(nlp.pipe(cleaned_texts, batch_size=batch_size_cpu, n_process=num_processes_for_spacy_cpu), total=len(cleaned_texts), desc="Лемматизация (CPU)"):
        lemmas = [token.lemma_ for token in doc]
        processed_lemmas.append(" ".join(lemmas))
    
    print(f"🔍 Предобработка завершена. Обработано {len(processed_lemmas)} текстов.")
    return processed_lemmas

# создаем функцию для получения усредненного эмбеддинга текста
def get_embeddings_batch(texts, tokenizer, model, device, batch_size=64):
    texts = list(texts)
    embeddings = []

    print(f"🔍 Начало генерации эмбеддингов для {len(texts)} текстов на устройстве '{device}'.")
    for i in tqdm(range(0, len(texts), batch_size), desc="Генерация эмбеддингов"):
        batch_texts = texts[i:i+batch_size]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        )

        # Переносим каждый тензор на устройство
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        # берем attention mask (1 — реальные токены, 0 — паддинг)
        mask = inputs["attention_mask"].unsqueeze(-1).expand(outputs.last_hidden_state.size())
        masked_embeddings = outputs.last_hidden_state * mask

        # считаем среднее только по непаддинговым токенам
        summed = masked_embeddings.sum(dim=1)
        counts = mask.sum(dim=1)
        mean_pooled = summed / counts

        embeddings.extend(mean_pooled.cpu().numpy())
    
    print(f"🔍 Генерация эмбеддингов завершена")
    
    return embeddings

# функция предобработки входного датасета
def prepare_data(data, is_train, scaler=None, ipca=None, 
                                 scaler_art=None, ipca_art=None,
                                 scaler_pro=None, ipca_pro=None,
                                 scaler_cou=None, ipca_cou=None):
    
    data = data.drop(
            [
                "robot_id",
                "accounts__id",
                "articles__id",
                "articles__user_id",
                "projects__id",
                "projects__user_id",
                "counterparties__id",
                "counterparties__user_id",
                "robots__user_id",
                "article_alternative_names__user_id",
            ],
            axis=1,
        )

    # поправим типы данных и заполним пропуски метками missing (для текстовых значений категорий) и 0 для пропущенных ID
    data[
        [
            "articles__parent_id",
            "projects__parent_id",
            "counterparties__parent_id",
            "robots__id",
        ]
    ] = (
        data[
            [
                "articles__parent_id",
                "projects__parent_id",
                "counterparties__parent_id",
                "robots__id",
            ]
        ]
        .fillna(0)
        .astype("int64")
    )

    data["purpose"] = data["purpose"].fillna("missing")
    data["articles__name"] = data["articles__name"].fillna("missing")
    data["projects__name"] = data["projects__name"].fillna("missing")
    data["counterparties__name"] = data["counterparties__name"].fillna("missing")

    # конвертируем дату в datetime
    data["date"] = pd.to_datetime(data["date"], yearfirst=True, errors='coerce')

    # и убираем записи из будущего и меньше нуля (и такое бывает)
    yesterday = pd.Timestamp.today().normalize() - pd.Timedelta(days=1)
    data = data[data["date"] <= yesterday]
    #data = data[data["payments_amount"] > 0]

    # переименуем и поправим тип столбца с фондами
    data = data.rename(columns={"accounts__user_id": "user_id"})
    data["user_id"] = data["user_id"].fillna(0).astype("int64")

    # кодируем текстовые поля
    # сначала очищаем и лемматизируем тексты
    data["clean_purpose"] = preprocess_texts_optimized(texts=data["purpose"],nlp_model_name="ru_core_news_sm")

    # грузим модели 
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained("DeepPavlov/rubert-base-cased")
    model = AutoModel.from_pretrained("DeepPavlov/rubert-base-cased")
    model = model.to(device)
    model.eval()
    
    # и запускаем генерацию эмбеддингов в назначении платежа
    data["purpose_emb"] = get_embeddings_batch(data["clean_purpose"], tokenizer, model, device)

    # 1. усредняем эмбеддинг в одно число
    data["purpose_mean"] = data["purpose_emb"].apply(lambda x: float(np.mean(x)))

    # 2. выделяем три главные компоненты с предварительным масштабированием и по батчам  
    batch_size = 10_000
    
    if is_train:
        scaler = StandardScaler()
        ipca = IncrementalPCA(n_components=3)

        # обучаем скейлер
        for i in tqdm(range(0, len(data), batch_size), desc="Обучение StandardScaler"):
            batch = np.vstack(data["purpose_emb"].iloc[i:i+batch_size])
            scaler.partial_fit(batch)

        # обучаем PCA на масштабированных данных
        for i in tqdm(range(0, len(data), batch_size), desc="Обучение IncrementalPCA"):
            batch = np.vstack(data["purpose_emb"].iloc[i:i+batch_size])
            batch_scaled = scaler.transform(batch)
            ipca.partial_fit(batch_scaled)

    # применяем трансформацию ко всему массиву
    transformed_batches = []
    for i in tqdm(range(0, len(data), batch_size), desc="Применяем PCA к эмбеддингам"):
        batch = np.vstack(data["purpose_emb"].iloc[i:i+batch_size]).astype(np.float32)
        batch_scaled = scaler.transform(batch)
        transformed_batches.append(ipca.transform(batch_scaled))
        
    purpose_pca_features = np.vstack(transformed_batches)

    # делаем датафрейм
    pca_column_names = [f"purpose_pca_{i+1}" for i in range(3)]
    data[pca_column_names] = purpose_pca_features

    # удалим ненужные столбцы
    data.drop(columns=["purpose", "clean_purpose", "purpose_emb"], inplace=True)

    # генерируем эмбеддинги для названий статей
    # сначала очищаем и лемматизируем тексты
    data["clean_articles__name"] = preprocess_texts_optimized(texts=data["articles__name"],nlp_model_name="ru_core_news_sm")
     # и запускаем генерацию эмбеддингов в названии статей
    data["articles__name_emb"] = get_embeddings_batch(data["clean_articles__name"], tokenizer, model, device)
    # усредняем эмбеддинг в одно число
    data["articles__name_mean"] = data["articles__name_emb"].apply(lambda x: float(np.mean(x)))
    
    # ----------------------------------------------------------
    #  Выделяем PCA (тоже 3 компоненты), полностью аналогично purpose
    # ----------------------------------------------------------

    batch_size = 10_000

    if is_train:
        scaler_art = StandardScaler()
        ipca_art = IncrementalPCA(n_components=3)

        # 1) обучаем scaler
        for i in tqdm(range(0, len(data), batch_size), desc="Обучение StandardScaler (articles)"):
            batch = np.vstack(data["articles__name_emb"].iloc[i:i+batch_size])
            scaler_art.partial_fit(batch)

        # 2) обучаем PCA
        for i in tqdm(range(0, len(data), batch_size), desc="Обучение IncrementalPCA (articles)"):
            batch = np.vstack(data["articles__name_emb"].iloc[i:i+batch_size])
            batch_scaled = scaler_art.transform(batch)
            ipca_art.partial_fit(batch_scaled)

    # 3) применяем трансформацию ко всем данным
    transformed_batches_art = []
    for i in tqdm(range(0, len(data), batch_size), desc="Применяем PCA к articles embeddings"):
        batch = np.vstack(data["articles__name_emb"].iloc[i:i+batch_size]).astype(np.float32)
        batch_scaled = scaler_art.transform(batch)
        transformed_batches_art.append(ipca_art.transform(batch_scaled))

    articles_pca_features = np.vstack(transformed_batches_art)

    # 4) создаём колонки
    art_pca_colnames = [f"articles_pca_{i+1}" for i in range(3)]
    data[art_pca_colnames] = articles_pca_features
    
    
    
    # удалим ненужные столбцы
    data.drop(columns=["articles__name", "clean_articles__name", "articles__name_emb"], inplace=True)

    # генерируем эмбеддинги для названий проектов
    # сначала очищаем и лемматизируем тексты
    data["clean_projects__name"] = preprocess_texts_optimized(texts=data["projects__name"],nlp_model_name="ru_core_news_sm")
     # и запускаем генерацию эмбеддингов в названии статей
    data["projects__name_emb"] = get_embeddings_batch(data["clean_projects__name"], tokenizer, model, device)
    # усредняем эмбеддинг в одно число
    data["projects__name_mean"] = data["projects__name_emb"].apply(lambda x: float(np.mean(x)))
    
    
     # ----------------------------------------------------------
    #  Выделяем PCA (тоже 3 компоненты), полностью аналогично purpose
    # ----------------------------------------------------------

    batch_size = 10_000

    if is_train:
        scaler_pro = StandardScaler()
        ipca_pro = IncrementalPCA(n_components=3)

        # 1) обучаем scaler
        for i in tqdm(range(0, len(data), batch_size), desc="Обучение StandardScaler (projects)"):
            batch = np.vstack(data["projects__name_emb"].iloc[i:i+batch_size])
            scaler_pro.partial_fit(batch)

        # 2) обучаем PCA
        for i in tqdm(range(0, len(data), batch_size), desc="Обучение IncrementalPCA (projects)"):
            batch = np.vstack(data["projects__name_emb"].iloc[i:i+batch_size])
            batch_scaled = scaler_pro.transform(batch)
            ipca_pro.partial_fit(batch_scaled)

    # 3) применяем трансформацию ко всем данным
    transformed_batches_pro = []
    for i in tqdm(range(0, len(data), batch_size), desc="Применяем PCA к projects embeddings"):
        batch = np.vstack(data["projects__name_emb"].iloc[i:i+batch_size]).astype(np.float32)
        batch_scaled = scaler_pro.transform(batch)
        transformed_batches_pro.append(ipca_pro.transform(batch_scaled))

    projects_pca_features = np.vstack(transformed_batches_pro)

    # 4) создаём колонки
    pro_pca_colnames = [f"projects_pca_{i+1}" for i in range(3)]
    data[pro_pca_colnames] = projects_pca_features
    
    
    
    # удалим ненужные столбцы
    data.drop(columns=["projects__name", "clean_projects__name", "projects__name_emb"], inplace=True)
    
    # генерируем эмбеддинги для названий доноров
    # сначала очищаем и лемматизируем тексты
    data["clean_counterparties__name"] = preprocess_texts_optimized(texts=data["counterparties__name"],nlp_model_name="ru_core_news_sm")
     # и запускаем генерацию эмбеддингов в названии статей
    data["counterparties__name_emb"] = get_embeddings_batch(data["clean_counterparties__name"], tokenizer, model, device)
    # усредняем эмбеддинг в одно число
    data["counterparties__name_mean"] = data["counterparties__name_emb"].apply(lambda x: float(np.mean(x)))
    
    
     # ----------------------------------------------------------
    #  Выделяем PCA (тоже 3 компоненты), полностью аналогично purpose
    # ----------------------------------------------------------

    batch_size = 10_000

    if is_train:
        scaler_cou = StandardScaler()
        ipca_cou = IncrementalPCA(n_components=3)

        # 1) обучаем scaler
        for i in tqdm(range(0, len(data), batch_size), desc="Обучение StandardScaler (counterparties)"):
            batch = np.vstack(data["counterparties__name_emb"].iloc[i:i+batch_size])
            scaler_cou.partial_fit(batch)

        # 2) обучаем PCA
        for i in tqdm(range(0, len(data), batch_size), desc="Обучение IncrementalPCA (counterparties)"):
            batch = np.vstack(data["counterparties__name_emb"].iloc[i:i+batch_size])
            batch_scaled = scaler_cou.transform(batch)
            ipca_cou.partial_fit(batch_scaled)

    # 3) применяем трансформацию ко всем данным
    transformed_batches_cou = []
    for i in tqdm(range(0, len(data), batch_size), desc="Применяем PCA к counterparties embeddings"):
        batch = np.vstack(data["counterparties__name_emb"].iloc[i:i+batch_size]).astype(np.float32)
        batch_scaled = scaler_cou.transform(batch)
        transformed_batches_cou.append(ipca_cou.transform(batch_scaled))

    counterparties_pca_features = np.vstack(transformed_batches_cou)

    # 4) создаём колонки
    cou_pca_colnames = [f"counterparties_pca_{i+1}" for i in range(3)]
    data[cou_pca_colnames] = counterparties_pca_features
    
    
    
    
    
    # удалим ненужные столбцы
    data.drop(columns=["counterparties__name", "clean_counterparties__name", "counterparties__name_emb"], inplace=True)


    # сбрасываем неактуальные столбцы
    data = data.drop(columns=[ 
        'date', 'expenditure',
        'payments_amount','user_id','account_id', 
        'contractor_id', 'article_id', 'project_id', 
        'counterpartie_id', 'donor_id', 'donor_cat_id',
        'articles__parent_id', 'projects__parent_id',
        'counterparties__parent_id', 'robots__id'])

    return data,scaler,ipca,scaler_art,ipca_art,scaler_pro,ipca_pro,scaler_cou,ipca_cou


### Загрузим и подготовим данные для прогноза

In [4]:
# загрузим данные скачанные при тесте 4, чтобы исключить последующих добавленных выписок/платежей

data_full = pd.read_csv('data_download.csv')

data_full.head(2)

,id,account_id,contractor_id,date,payments_amount,purpose,article_id,expenditure,project_id,counterpartie_id,donor_id,robot_id,donor_cat_id,accounts__id,accounts__user_id,articles__id,articles__user_id,articles__parent_id,articles__name,projects__id,projects__user_id,projects__parent_id,projects__name,counterparties__id,counterparties__user_id,counterparties__parent_id,counterparties__name,robots__id,robots__user_id,article_alternative_names__user_id,uc__uc_id
0,289,1,70,2022-09-08,1.0,NaN,0,incoming,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,290,1,71,2022-09-08,1.0,NaN,0,incoming,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# далее проведем отбор нужных для теста строк аналогично тесту 4 (см.там подробнее)

# отсеиваем платежи, участвовавшие в последнем обучении, по id
train_history = pd.read_csv('train_history.csv')
last_train_payment_id = train_history['last_payment_id'].iloc[-1]

data_full = data_full[data_full['id']>last_train_payment_id].copy()
data_full['date'] = pd.to_datetime(data_full['date'], errors='coerce')

display(data_full.id.count(),data_full.uc__uc_id.isna().sum())


data_full = data_full.dropna(subset=['articles__name', 'projects__name', 'counterparties__name'], how='all') # здесь нет прогнозов изза отсутствия основных признаков
data_full = data_full.dropna(subset=['uc__uc_id']) # здесь нет прогнозов каким-то причинам - метки не загружались в связи сдоработкой модели
data_full = data_full.dropna(subset=['articles__name']) #сбраываем строки где нет значения статьи, мы не сможем истинные значения расставаить по этим строкам в соответствии со словарем

display(data_full.id.count())

main_dict = pd.read_csv('main_dict.csv')

data_full['articles__name'] = data_full['articles__name'].str.lower()

# добавляем истинные метки новым платежам по основному словарю, которы участвовали в разметке тренировочной выборки
data_new_main_articles = data_full.merge(
    main_dict[['raw', 'uc_id']],
    left_on='articles__name',
    right_on='raw',
    how='left'
)
data_new_main_articles.rename(columns={'uc_id': 'target'}, inplace=True)
data_new_main_articles.drop(columns=['raw'], inplace=True)

# выбираем платежи с отсутстивующими истинными метками - для теуущего теста
data_new_staging_articles = data_new_main_articles[data_new_main_articles.target.isna()].copy()

# сбрасывем эти платежи из основого массива с метками по старому словарю
data_new_main_articles = data_new_main_articles.dropna(subset=['target'])

display(data_new_main_articles.id.count(),data_new_staging_articles.id.count())


60114

13069

46532

44811

1721

In [6]:
# дале добавим истинные метки в тестовый датасет
staging_dict = pd.read_csv('staging_dict.csv')

data_new_staging_articles.drop(columns=['uc__uc_id','target'], inplace=True)

data_new_staging_articles = data_new_staging_articles.merge(
    staging_dict[['raw', 'uc_id']],
    left_on='articles__name',
    right_on='raw',
    how='left'
)

data_new_staging_articles.rename(columns={'uc_id': 'target'}, inplace=True)
data_new_staging_articles.drop(columns=['raw'], inplace=True)
data_new_staging_articles.head(3)

,id,account_id,contractor_id,date,payments_amount,purpose,article_id,expenditure,project_id,counterpartie_id,donor_id,robot_id,donor_cat_id,accounts__id,accounts__user_id,articles__id,articles__user_id,articles__parent_id,articles__name,projects__id,projects__user_id,projects__parent_id,projects__name,counterparties__id,counterparties__user_id,counterparties__parent_id,counterparties__name,robots__id,robots__user_id,article_alternative_names__user_id,target
0,1102148,2881,41747,2025-10-09,29.00,"//Приложение//кол-во распор. 1 //Перевод ср-в согл. реестра переводов за 08.10.2025 по Дог. 560/2017 от 07.12.2017, ...",48528,incoming,3801,0,0,13320,9985,2881.0,962.0,48528.0,962.0,41226.0,мкб_терминал,3801.0,962.0,0.0,Уставные деятельность,9985.0,962.0,9982.0,...массовые,13320.0,962.0,NaN,1
1,1102255,114,75,2025-10-09,2605.00,Перевод денежных средств по договору присоединения к условиям оказания услуг ИТО при осуществлении переводов денежны...,49607,incoming,1980,0,0,-1,2092,114.0,187.0,49607.0,187.0,47651.0,вк добро физлица,1980.0,187.0,0.0,Уставные цели,2092.0,187.0,13621.0,ВК добро,NaN,NaN,NaN,2
2,1102256,114,46,2025-10-09,13736.01,Перечисление денежных средств по Договору №ПД-1299 от 2024-01-30 по платежам за 2025-10-08. DS.2471242. Сумма удержа...,49853,incoming,1980,0,0,-1,12584,114.0,187.0,49853.0,187.0,47651.0,тбанк физлица,1980.0,187.0,0.0,Уставные цели,12584.0,187.0,13621.0,ТБанк,NaN,NaN,NaN,1


In [8]:
# предобрабатываем датасет
scaler_emb_path = 'scaler.pkl'
ipca_emb_path = 'ipca.pkl'
scaler_art_emb_path = 'scaler_art.pkl'
ipca_art_emb_path = 'ipca_art.pkl'
scaler_pro_emb_path = 'scaler_pro.pkl'
ipca_pro_emb_path = 'ipca_pro.pkl'
scaler_cou_emb_path = 'scaler_cou.pkl'
ipca_cou_emb_path = 'ipca_cou.pkl'


scaler = joblib.load(scaler_emb_path)
ipca = joblib.load(ipca_emb_path)
scaler_art = joblib.load(scaler_art_emb_path)
ipca_art = joblib.load(ipca_art_emb_path)
scaler_pro = joblib.load(scaler_pro_emb_path)
ipca_pro = joblib.load(ipca_pro_emb_path)
scaler_cou = joblib.load(scaler_cou_emb_path)
ipca_cou = joblib.load(ipca_cou_emb_path)

dnsa_prepared,_,_,_,_,_,_,_,_ = prepare_data(data_new_staging_articles, is_train=False, scaler=scaler, ipca=ipca,
                                                                scaler_art=scaler_art, ipca_art=ipca_art,
                                                                scaler_pro=scaler_pro, ipca_pro=ipca_pro,
                                                                scaler_cou=scaler_cou, ipca_cou=ipca_cou)


🔍 Запуск предобработки для 1721 текстов...


Очистка: 100%|██████████| 1721/1721 [00:00<00:00, 262029.81it/s]

🔍 Загрузка spaCy модели: 'ru_core_news_sm'.


🔍 Лемматизация будет выполнена в 7 потоках.


Лемматизация (CPU): 100%|██████████| 1721/1721 [00:22<00:00, 76.39it/s] 


...

Лемматизация (CPU): 100%|██████████| 1721/1721 [00:20<00:00, 83.48it/s]


🔍 Предобработка завершена. Обработано 1721 текстов.
🔍 Начало генерации эмбеддингов для 1721 текстов на устройстве 'cpu'.


Генерация эмбеддингов:   0%|          | 0/27 [00:00<?, ?it/s]/opt/anaconda3/envs/datasphere_cli_env/lib/python3.10/site-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
Генерация эмбеддингов: 100%|██████████| 27/27 [00:04<00:00,  6.13it/s]


🔍 Генерация эмбеддингов завершена


Применяем PCA к counterparties embeddings: 100%|██████████| 1/1 [00:00<00:00, 67.21it/s]


In [9]:
dnsa_prepared.head(3)

,id,target,purpose_mean,purpose_pca_1,purpose_pca_2,purpose_pca_3,articles__name_mean,articles_pca_1,articles_pca_2,articles_pca_3,projects__name_mean,projects_pca_1,projects_pca_2,projects_pca_3,counterparties__name_mean,counterparties_pca_1,counterparties_pca_2,counterparties_pca_3
0,1102148,1,-0.001918,9.695052,11.319215,-3.510300,-0.000515,8.515697,2.002144,-0.050886,-0.001267,-42.619823,-6.999869,-1.146760,-0.000312,-14.620590,-5.308704,-2.298226
1,1102255,2,0.000909,13.736932,-5.440735,-9.585536,0.000781,11.291774,-2.483096,3.839670,0.001069,5.499426,6.390220,-5.007211,-0.000621,-0.760723,3.053576,-3.153119
2,1102256,1,-0.001504,16.288244,4.061973,-14.460937,-0.000501,9.526849,2.485729,-0.767988,0.001069,5.499426,6.390220,-5.007211,-0.000001,-0.617364,4.125260,-2.304470


In [10]:
# разделим признаки
X_data_prepared = dnsa_prepared.drop(columns=['id','target'])
y_data_prepared = dnsa_prepared['target']



### Генерируем прогнозы и считаем метрики

#### catboost

In [11]:
best_model_cb = CatBoostClassifier()
best_model_cb.load_model("model_cb.cbm")
print(f"Параметры загруженной модели:\n{best_model_cb.get_params()}")

# предсказания catboost
y_pred_cb = best_model_cb.predict(X_data_prepared)

# метрики
accuracy = accuracy_score(y_data_prepared, y_pred_cb)
precision = precision_score(y_data_prepared, y_pred_cb, average='weighted')
recall = recall_score(y_data_prepared, y_pred_cb, average='macro')
f1 = f1_score(y_data_prepared, y_pred_cb, average='weighted')

y_proba_cb = best_model_cb.predict_proba(X_data_prepared)
y_proba_cb = y_proba_cb / y_proba_cb.sum(axis=1, keepdims=True) # при обучении модели с функцией MultiClassOneVsAll
roc_auc = roc_auc_score(y_data_prepared, y_proba_cb, multi_class='ovr')

print("Accuracy:", accuracy)
print("Precision (weighted):", precision)
print("Recall (macro):", recall)
print("F1-score (weighted):", f1)
print("ROC-AUC:", roc_auc)

print("\nОтчет по классам:")
print(classification_report(y_data_prepared, y_pred_cb))

Параметры загруженной модели:
{'use_best_model': True, 'random_strength': 5, 'border_count': 256, 'od_wait': 10, 'verbose': 100, 'iterations': 1000, 'auto_class_weights': 'SqrtBalanced', 'loss_function': 'MultiClassOneVsAll', 'l2_leaf_reg': 9, 'task_type': 'GPU', 'depth': 6, 'min_data_in_leaf': 1, 'learning_rate': 0.05, 'random_seed': 42}
Accuracy: 0.8576409064497386
Precision (weighted): 0.8684953796037186
Recall (macro): 0.47910592347284264
F1-score (weighted): 0.8543911761596121
ROC-AUC: 0.8901488598199957

Отчет по классам:
              precision    recall  f1-score   support

           1       0.90      0.98      0.94      1267
           2       0.88      0.58      0.70       388
           3       0.12      0.25      0.16         8
           4       1.00      1.00      1.00         2
           5       0.00      0.00      0.00        34
           6       0.00      0.00      0.00        10
           7       1.00      0.50      0.67         8
           8       1.00      1.00

/opt/anaconda3/envs/datasphere_cli_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/anaconda3/envs/datasphere_cli_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/anaconda3/envs/datasphere_cli_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(re